# mel_sandbox_data_3: How much does each exclusion rule remove?

Data task, item 3 (exclusions). For every rule we measure two things:

- **Entries**: cells of `n1_peak_sample` (recording channel x stimulation pair) that the rule removes.
- **N1 responses**: entries with a detected N1 that the rule removes.

Earlier numbers suggested that the *bad channel* and *no label* rules remove many entries but no N1s,
while the *13 mm*, *resected* and *SOZ* rules remove most N1s. This notebook checks that, and also
checks how much the rules overlap.

Only recording channels are checked here (not the stimulated electrodes).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.paths import load_ccep, runs, electrodes_tsv, channels_tsv

EXCLUSION_RADIUS_MM = 13

cc = load_ccep()
print(len(cc), "patients")

## 1. Two facts about the data this notebook relies on

- `nativeElecDistances` (one matrix per patient) is in the **row order of electrodes.tsv**, not in the
  order of `channel_names`. Checked: its size always equals the number of rows in electrodes.tsv, and it
  correlates ~0.96 with distances computed from the MNI coordinates in that order. In 8 patients the
  channel list differs from electrodes.tsv (extra or missing channels), so we must match by name.
- Stimulation pairs come as names like `"PT03-PT02"` in `stimpair_names`.
- Electrodes with `group == "other"` are non-brain channels (ECG, markers, unused inputs). They have no
  hemisphere or atlas label, which is why the "no label" rule removes so many entries.

## 2. Build one table with one row per entry

For each entry we store whether it has an N1, the distance from the recording electrode to the
**closer** of the two stimulated electrodes, and one True/False column per rule.

In [ ]:
def norm(name):
    return str(name).strip().upper()


rows = []
for p in cc:
    el = electrodes_tsv(p["id"], p["ses"])
    el["key"] = el["name"].map(norm)
    el_index = {k: i for i, k in enumerate(el["key"])}   # row in electrodes.tsv = row in nativeElecDistances
    el_by_key = el.set_index("key")
    D = np.asarray(p["nativeElecDistances"], dtype=float)
    assert D.shape == (len(el), len(el)), p["id"]

    for r_i, r in enumerate(runs(p)):
        ch_names = [norm(c) for c in r["channel_names"]]
        n1 = np.asarray(r["n1_peak_sample"], dtype=float)          # channels x stimulation pairs
        pairs = [tuple(norm(x) for x in str(s).split("-")) for s in r["stimpair_names"]]
        assert n1.shape == (len(ch_names), len(pairs)), p["id"]

        # Distance to the closer stimulated electrode (NaN if a name is not in electrodes.tsv)
        ch_i = np.array([el_index.get(c, -1) for c in ch_names])
        a_i = np.array([el_index.get(a, -1) for a, b in pairs])
        b_i = np.array([el_index.get(b, -1) for a, b in pairs])
        D_pad = np.vstack([np.hstack([D, np.full((len(D), 1), np.nan)]), np.full((1, len(D) + 1), np.nan)])
        dist = np.fmin(D_pad[np.ix_(ch_i, a_i)], D_pad[np.ix_(ch_i, b_i)])   # index -1 -> NaN row/col

        ch_tsv = channels_tsv(p["id"], p["ses"], r["runName"])
        bad = set(ch_tsv.loc[ch_tsv["status"].astype(str).str.lower() == "bad", "name"].map(norm))

        info = el_by_key.reindex(ch_names)
        ch_flags = pd.DataFrame({
            "channel": ch_names,
            "missing_tsv": info["name"].isna().to_numpy(),
            "depth": (info["group"] == "depth").to_numpy(),
            "no_label": (info["hemisphere"].isna() | info["Destrieux_label"].isna()).to_numpy(),
            "label0": (info["Destrieux_label"] == 0).to_numpy(),
            "bad": np.isin(ch_names, list(bad)),
            "resected": (info["resected"] == "yes").to_numpy(),
            "soz": (info["soz"] == "yes").to_numpy(),
        })

        n_ch, n_pairs = n1.shape
        run_df = ch_flags.loc[np.repeat(np.arange(n_ch), n_pairs)].reset_index(drop=True)
        run_df.insert(0, "patient", p["id"])
        run_df.insert(1, "run", r_i)
        run_df["pair"] = np.tile(np.arange(n_pairs), n_ch)
        run_df["has_n1"] = ~np.isnan(n1.ravel())
        run_df["dist_mm"] = dist.ravel()
        rows.append(run_df)

df = pd.concat(rows, ignore_index=True)
df["near_stim"] = df["dist_mm"] < EXCLUSION_RADIUS_MM

print(f"{len(df):,} entries, {df['has_n1'].sum():,} with an N1, {df['patient'].nunique()} patients")
print(f"Entries without a distance: {df['dist_mm'].isna().sum():,} "
      f"({df.loc[df['dist_mm'].isna(), 'has_n1'].sum():,} of them with an N1)")

## 3. Each rule on its own

- **% of entries / % of N1s**: what the rule removes if we look at it alone.
- **% of N1s, only this rule**: N1s that *no other rule* would remove. If this is much smaller
  than the column before, the rule mostly overlaps with other rules.

In [ ]:
RULES = {
    f"Within {EXCLUSION_RADIUS_MM} mm of a stimulated electrode": "near_stim",
    "Depth electrode": "depth",
    "No hemisphere / atlas label": "no_label",
    "Atlas label 0": "label0",
    "Bad channel (channels.tsv)": "bad",
    "Resected": "resected",
    "Seizure onset zone (SOZ)": "soz",
    "Missing from electrodes.tsv": "missing_tsv",
}
flag_cols = list(RULES.values())
n_rules_hit = df[flag_cols].sum(axis=1)
n1_total = df["has_n1"].sum()

summary = pd.DataFrame({
    "% of entries": [100 * df[c].mean() for c in flag_cols],
    "% of N1s": [100 * (df[c] & df["has_n1"]).sum() / n1_total for c in flag_cols],
    "% of N1s, only this rule": [
        100 * (df[c] & df["has_n1"] & (n_rules_hit == 1)).sum() / n1_total for c in flag_cols
    ],
}, index=list(RULES.keys())).round(1)

kept = n_rules_hit == 0
summary.loc["Kept after all rules"] = [
    round(100 * kept.mean(), 1),
    round(100 * (kept & df["has_n1"]).sum() / n1_total, 1),
    np.nan,
]
summary

## 4. Why do bad channels and unlabeled channels have no N1s?

Two possible reasons:
1. The N1 detection already skipped these channels (all their values are NaN).
2. They are the same channels (e.g. non-brain channels such as ECG, which are both bad and unlabeled).

The cross-table shows how much the two rules overlap.

In [ ]:
for col, label in [("bad", "Bad channels"), ("no_label", "Unlabeled channels")]:
    sub = df[df[col]]
    print(f"{label}: {len(sub):,} entries, {sub['has_n1'].sum():,} with an N1")

print()
pd.crosstab(df["bad"], df["no_label"], margins=True).rename_axis(index="bad", columns="no_label")

## 5. Applying the rules one after another

The numbers in section 4 overlap, so they do not add up. Here the rules are applied in order, and we
see how many N1s are left after each step. The order matters for how much each step removes,
but not for the final result.

In [ ]:
order = ["missing_tsv", "bad", "no_label", "label0", "depth", "resected", "soz", "near_stim"]
names = {v: k for k, v in RULES.items()}

remaining = np.ones(len(df), dtype=bool)
steps = [("All entries", remaining.sum(), (remaining & df["has_n1"]).sum())]
for col in order:
    remaining &= ~df[col].to_numpy()
    steps.append((f"- {names[col]}", remaining.sum(), (remaining & df["has_n1"]).sum()))

waterfall = pd.DataFrame(steps, columns=["Step", "Entries left", "N1s left"])
waterfall["% of N1s left"] = (100 * waterfall["N1s left"] / n1_total).round(1)
waterfall

## 6. Plots

**Left:** share of all N1s that each rule removes on its own.
**Right:** how often an N1 is detected at each distance from the stimulated electrodes. This shows
why the 13 mm rule removes so many N1s: close electrodes almost always "respond", partly because
of volume conduction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.1, 1]})
color = "#3b6fb6"

# Left: % of N1s removed per rule
s = summary.drop("Kept after all rules")["% of N1s"].sort_values()
ax = axes[0]
ax.barh(s.index, s.values, color=color, height=0.6)
for y, v in enumerate(s.values):
    ax.text(v + 0.5, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlabel("% of all N1 responses removed (rule on its own)")
ax.set_title("Which rules remove N1 responses?")
ax.spines[["top", "right"]].set_visible(False)

# Right: N1 rate by distance (labelled channels with a known distance only)
ok = df["dist_mm"].notna() & ~df["bad"] & ~df["no_label"]
bins = np.arange(0, 82, 2)
binned = df.loc[ok].groupby(pd.cut(df.loc[ok, "dist_mm"], bins), observed=True)["has_n1"].agg(["mean", "size"])
binned = binned[binned["size"] >= 50]  # skip distance bins with too few entries to be reliable
centers = [b.mid for b in binned.index]
ax = axes[1]
ax.plot(centers, 100 * binned["mean"].values, color=color, lw=2, marker="o", ms=4)
ax.axvline(EXCLUSION_RADIUS_MM, color="grey", ls="--", lw=1)
ax.text(EXCLUSION_RADIUS_MM + 1, ax.get_ylim()[1] * 0.92, f"{EXCLUSION_RADIUS_MM} mm", color="grey")
ax.set_xlabel("Distance to the closer stimulated electrode (mm)")
ax.set_ylabel("Entries with an N1 (%)")
ax.set_title("N1 detection rate by distance")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. SOZ and resection: which patients have them, and how they overlap

Venn diagrams: each circle is one group, its area is proportional to its size, and the shared part
is the overlap. The number of patients / electrodes in **neither** group is written under each diagram.

- **Left (patients):** patients with at least one electrode marked as SOZ, or as resected.
  "Neither" can mean the annotation was not done, or there really was no SOZ / no surgery.
- **Right (electrodes):** brain electrodes only (group "other" left out), across all patients.

Needs the `matplotlib-venn` package: `pip install matplotlib-venn`.

In [ ]:
from matplotlib_venn import venn2

SOZ_COLOR = "#3b6fb6"        # blue
RESECTED_COLOR = "#e0812c"   # orange; the overlap shows as the mix of both

el_all = pd.concat(
    [electrodes_tsv(p["id"], p["ses"]).assign(patient=p["id"]) for p in cc], ignore_index=True
)
el_all["is_soz"] = el_all["soz"] == "yes"
el_all["is_resected"] = el_all["resected"] == "yes"

per_patient = el_all.groupby("patient")[["is_soz", "is_resected"]].any()
brain = el_all[el_all["group"] != "other"]


def overlap_counts(flags):
    # (only SOZ, only resected, both, neither) - the order venn2 expects for the first three
    s, r = flags["is_soz"], flags["is_resected"]
    return (s & ~r).sum(), (~s & r).sum(), (s & r).sum(), (~s & ~r).sum()


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, flags, unit, title in [
    (axes[0], per_patient, "patients", f"Patients (n = {len(per_patient)})"),
    (axes[1], brain, "electrodes", f"Brain electrodes (n = {len(brain):,})"),
]:
    only_s, only_r, both, neither = overlap_counts(flags)
    n = len(flags)
    v = venn2(subsets=(only_s, only_r, both), set_labels=("", ""),
              set_colors=(SOZ_COLOR, RESECTED_COLOR), alpha=0.6, ax=ax)
    for region, count in [("10", only_s), ("01", only_r), ("11", both)]:
        label = v.get_label_by_id(region)
        if label is not None:
            label.set_text(f"{count:,}\n({100 * count / n:.0f}%)")
            label.set_fontsize(10)
    ax.set_title(f"{title}\nNeither SOZ nor resected: {neither:,} ({100 * neither / n:.0f}%)", fontsize=12)
    print(f"{unit}: only SOZ {only_s}, only resected {only_r}, both {both}, neither {neither}")

from matplotlib.patches import Patch
overlap_color = v.get_patch_by_id("11").get_facecolor()
fig.legend(handles=[Patch(color=SOZ_COLOR, alpha=0.6, label="SOZ only"),
                    Patch(color=overlap_color, label="SOZ and resected"),
                    Patch(color=RESECTED_COLOR, alpha=0.6, label="Resected only")],
           loc="lower center", ncol=3, frameon=False, fontsize=11)
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

The same numbers as shares of the **whole** set (100% bars), so the "neither" group is visible too.
Segments under 3% are not labelled; their numbers are printed above.

In [ ]:
NEITHER_COLOR = "#f0f0f0"
segments = [
    ("SOZ only", SOZ_COLOR, 0.6),
    ("SOZ and resected", overlap_color, None),  # keep the Venn overlap colour as is
    ("Resected only", RESECTED_COLOR, 0.6),
    ("Neither", NEITHER_COLOR, 1.0),
]
rows = [("Patients", per_patient), ("Brain electrodes", brain)]

fig, ax = plt.subplots(figsize=(12, 2.8))
for y, (row_name, flags) in enumerate(rows):
    only_s, only_r, both, neither = overlap_counts(flags)
    counts = [only_s, both, only_r, neither]
    n = len(flags)
    left = 0.0
    for (label, seg_color, seg_alpha), count in zip(segments, counts):
        share = 100 * count / n
        ax.barh(y, share, left=left, color=seg_color, alpha=seg_alpha, height=0.6,
                edgecolor="white", linewidth=2, label=label if y == 0 else None)
        if share >= 3:
            ax.text(left + share / 2, y, f"{share:.0f}%\n({count:,})", ha="center", va="center", fontsize=9)
        left += share

ax.set_yticks(range(len(rows)))
ax.set_yticklabels([f"{name}\n(n = {len(flags):,})" for name, flags in rows])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("% of all patients / all brain electrodes")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=4, frameon=False)
plt.tight_layout()
plt.show()

## 8. Does removing SOZ electrodes change the observed spread?

Test in the 36 patients **with** an SOZ annotation: for every stimulation site, compute the spread
(share of recording electrodes with an N1) twice:

- **with SOZ electrodes**: as if we did not know where the SOZ is (the situation in the unannotated patients),
- **without SOZ electrodes**: SOZ removed.

All other exclusion rules (including resected) are applied in both. If the two values are almost the
same for most sites, leaving unknown SOZ electrodes in the unannotated patients should change little.

Spread here is computed per electrode, not per region, so the numbers will shift a bit after the
region assignment.

In [ ]:
annotated_patients = per_patient.index[per_patient["is_soz"]]
other_rules = ["missing_tsv", "bad", "no_label", "label0", "depth", "near_stim", "resected"]
kept = df[~df[other_rules].any(axis=1) & df["patient"].isin(annotated_patients)]

site = ["patient", "run", "pair"]
spread = pd.DataFrame({
    "with_soz": kept.groupby(site)["has_n1"].mean(),
    "without_soz": kept[~kept["soz"]].groupby(site)["has_n1"].mean(),
}).dropna()
spread["diff"] = spread["with_soz"] - spread["without_soz"]

r = spread["with_soz"].corr(spread["without_soz"])
abs_diff = spread["diff"].abs()
print(f"Stimulation sites: {len(spread):,}")
print(f"Correlation with vs without SOZ: r = {r:.3f}")
print(f"Sites with no change at all: {100 * (abs_diff == 0).mean():.0f}%")
print(f"Absolute change: median {100 * abs_diff.median():.1f}, 95th percentile {100 * abs_diff.quantile(0.95):.1f}, "
      f"max {100 * abs_diff.max():.1f} percentage points")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

ax = axes[0]
ax.scatter(100 * spread["without_soz"], 100 * spread["with_soz"], s=10, alpha=0.4, color=color, edgecolor="none")
ax.plot([0, 100], [0, 100], color="grey", ls="--", lw=1, label="no change (identity line)")
lim = 100 * spread[["with_soz", "without_soz"]].max().max() * 1.05
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal")
ax.set_xlabel("Spread without SOZ electrodes (%)")
ax.set_ylabel("Spread with SOZ electrodes (%)")
ax.set_title(f"Each dot = one stimulation site (n = {len(spread):,}), r = {r:.3f}")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.hist(100 * spread["diff"], bins=np.arange(-13, 13.5, 0.5), color=color)
ax.axvline(0, color="grey", ls="--", lw=1)
ax.set_xlabel("Change in spread when SOZ electrodes are kept (percentage points)")
ax.set_ylabel("Number of stimulation sites")
ax.set_title("How much does the spread change?")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 9. What to look for

- Is "% of N1s" really ~0 for **bad channel** and **no label**? Then these rules do not change the
  results, they only clean the table.
- Do **bad** and **no label** mostly hit the same channels (section 4)?
- How much of the **resected** effect is "only this rule"? Resected and SOZ electrodes probably overlap.
- Section 7: 33 of 74 patients have neither marking. Excluding SOZ is only possible in the annotated
  patients, which is why we keep the robustness check with only those patients.
- In the distance plot: does the N1 rate drop clearly around 13 mm, or does it keep falling
  smoothly? A smooth curve means the exact radius is somewhat arbitrary, which is a good reason to
  vary it in the robustness analysis (e.g. 10 / 13 / 20 mm).